In [1]:
import torch
from torch import nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
import torch.optim as optim
from torchinfo import summary

In [2]:
device = ("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using {device} device")

Using cuda device


In [16]:
#잔차블록 클래스 
class ResidualBlock (nn.Module):
    def __init__(self, in_channels, filters, first_stride=1, conv_skip=None):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=filters, kernel_size=1, stride=first_stride)
        self.bn1 = nn.BatchNorm2d(filters, eps=1e-5)
        self.relu1 = nn.ReLU()

        self.conv2 = nn.Conv2d(in_channels=filters, out_channels=filters, kernel_size=3, padding='same')
        self.bn2 = nn.BatchNorm2d(filters, eps=1e-5)
        self.relu2 = nn.ReLU()

        self.conv3 = nn.Conv2d(in_channels=filters, out_channels=filters * 4, kernel_size=1)
        self.bn3 = nn.BatchNorm2d(filters * 4, eps=1e-5)
        self.relu3 = nn.ReLU()
        self.conv_skip = conv_skip #스킵연결시 shape맞추는 변수(첫번째 잔차블록,스택에서만 적용되어야함)

    def forward(self, x):
        skip_conn = x  
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)

        x = self.conv3(x)
        x = self.bn3(x)

        # 첫번째 잔차블록에서만 True로 실행 
        if self.conv_skip is not None: 
            skip_conn = self.conv_skip(skip_conn)

        x = x + skip_conn
        x = self.relu3(x)
        
        return x

In [23]:
#전체모델 연결 클래스
class ResNet50(nn.Module):
    
    #잔차 스택 구성 함수
    def residual_stack(self, in_channels, blocks, filters, first_stride=2):
        layers = []

        conv_skip = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=filters * 4, kernel_size=1, stride=first_stride),
            nn.BatchNorm2d(filters * 4, eps=1e-5)
        )
        
        #첫번째 블록
        layers.append(
            ResidualBlock(in_channels=in_channels, filters=filters, first_stride=first_stride, conv_skip=conv_skip)
        )
        
        #첫번째 이후 블록
        for _ in range(1, blocks):
            layers.append(
                ResidualBlock(in_channels=filters * 4, filters=filters, first_stride=1, conv_skip=None)
            )

        #쌓은 레이어를 전부 반환해줘야 함
        return nn.Sequential(*layers)

    def __init__(self):
        super().__init__()
        #입력층
        self.features = nn.Sequential(
            nn.ZeroPad2d(padding=3),
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, stride=2), 
            nn.BatchNorm2d(num_features=64, eps=1e-5),
            nn.ReLU(),
            nn.ZeroPad2d(padding=1),
            nn.MaxPool2d(kernel_size=3, stride=2)
        )

        #잔차스택 1,2,3,4번째
        self.layer1 = self.residual_stack(in_channels=64, blocks=3, filters=64, first_stride=1)
        self.layer2 = self.residual_stack(in_channels=256, blocks=4, filters=128, first_stride=2)
        self.layer3 = self.residual_stack(in_channels=512, blocks=6, filters=256, first_stride=2)
        self.layer4 = self.residual_stack(in_channels=1024, blocks=3, filters=512, first_stride=2)

        #출력층
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()
        self.full_connect = nn.Linear(2048, 1000)

    
    # 실제 인-아웃 연결 
    def forward(self, x):
        x = self.features(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = self.flatten(x)
        x = self.full_connect(x)

        return x

In [18]:
resnet50 = ResNet50().to(device)
summary(resnet50, input_size=(32,3,224,224))

Layer (type:depth-idx)                   Output Shape              Param #
ResNet50                                 [32, 1000]                --
├─Sequential: 1-1                        [32, 64, 56, 56]          --
│    └─ZeroPad2d: 2-1                    [32, 3, 230, 230]         --
│    └─Conv2d: 2-2                       [32, 64, 112, 112]        9,472
│    └─BatchNorm2d: 2-3                  [32, 64, 112, 112]        128
│    └─ReLU: 2-4                         [32, 64, 112, 112]        --
│    └─ZeroPad2d: 2-5                    [32, 64, 114, 114]        --
│    └─MaxPool2d: 2-6                    [32, 64, 56, 56]          --
├─Sequential: 1-2                        [32, 256, 56, 56]         --
│    └─ResidualBlock: 2-7                [32, 256, 56, 56]         --
│    │    └─Conv2d: 3-1                  [32, 64, 56, 56]          4,160
│    │    └─BatchNorm2d: 3-2             [32, 64, 56, 56]          128
│    │    └─ReLU: 3-3                    [32, 64, 56, 56]          --
│    │ 